# GFS要素清单：454-471

In [ ]:
from reki import from_source

In [ ]:
from data_notebook.data import get_data_file

gfs_grib2_orig_file_path = get_data_file("cma-gfs")

examples = {}
gfs_grib2_orig_file_path

## 土壤温度

| 要素名 | cemc | wgrib2 | eccodes | GRIB Key | 层次类型 | 层次 | 编号 |
|---|---|---|---|---|---|---|---|
| 土壤温度 | st(0-10) | TMP | t | (0, 0, 0) | depthBelowLandLayer | 0, 0.1 | 464 |
| 土壤温度 | st(10-40) | TMP | t | (0, 0, 0) | depthBelowLandLayer | 0.1, 0.4 | 465 |
| 土壤温度 | st(40-100) | TMP | t | (0, 0, 0) | depthBelowLandLayer | 0.4, 1 | 466 |
| 土壤温度 | st(100-200) | TMP | t | (0, 0, 0) | depthBelowLandLayer | 1, 2 | 467 |

In [ ]:
examples[464] = (
    {"parameter": "cedarkit.st-0-10"},
    {
        "discipline": 0,
        "parameterCategory": 0,
        "parameterNumber": 0,
        "productDefinitionTemplateNumber": 0,
        "typeOfFirstFixedSurface": 106,
        "typeOfSecondFixedSurface": 106,
        "scaledValueOfFirstFixedSurface": 0,
    },
)

In [ ]:
examples[465] = (
    {"parameter": "cedarkit.st-10-40"},
    {
        "discipline": 0,
        "parameterCategory": 0,
        "parameterNumber": 0,
        "productDefinitionTemplateNumber": 0,
        "typeOfFirstFixedSurface": 106,
        "typeOfSecondFixedSurface": 106,
        "scaledValueOfFirstFixedSurface": 10,
    },
)

In [ ]:
examples[466] = (
    {"parameter": "cedarkit.st-40-100"},
    {
        "discipline": 0,
        "parameterCategory": 0,
        "parameterNumber": 0,
        "productDefinitionTemplateNumber": 0,
        "typeOfFirstFixedSurface": 106,
        "typeOfSecondFixedSurface": 106,
        "scaledValueOfFirstFixedSurface": 40,
    },
)

In [ ]:
examples[467] = (
    {"parameter": "cedarkit.st-100-200"},
    {
        "discipline": 0,
        "parameterCategory": 0,
        "parameterNumber": 0,
        "productDefinitionTemplateNumber": 0,
        "typeOfFirstFixedSurface": 106,
        "typeOfSecondFixedSurface": 106,
        "scaledValueOfFirstFixedSurface": 100,
    },
)

## 土壤湿度

| 要素名 | cemc | wgrib2 | eccodes | GRIB Key | 层次类型 | 层次 | 编号 |
|---|---|---|---|---|---|---|---|
| 土壤湿度 | sw(0-10) | SPFH | q | (0, 1, 0) | depthBelowLandLayer | 0, 0.1 | 468 |
| 土壤湿度 | sw(10-40) | SPFH | q | (0, 1, 0) | depthBelowLandLayer | 0.1, 0.4 | 469 |
| 土壤湿度 | sw(40-100) | SPFH | q | (0, 1, 0) | depthBelowLandLayer | 0.4, 1 | 470 |
| 土壤湿度 | sw(100-200) | SPFH | q | (0, 1, 0) | depthBelowLandLayer | 1, 2 | 471 |

In [ ]:
examples[468] = (
    {"parameter": "cedarkit.sw-0-10"},
    {
        "discipline": 0,
        "parameterCategory": 1,
        "parameterNumber": 0,
        "productDefinitionTemplateNumber": 0,
        "typeOfFirstFixedSurface": 106,
        "typeOfSecondFixedSurface": 106,
        "scaledValueOfFirstFixedSurface": 0,
    },
)

In [ ]:
examples[469] = (
    {"parameter": "cedarkit.sw-10-40"},
    {
        "discipline": 0,
        "parameterCategory": 1,
        "parameterNumber": 0,
        "productDefinitionTemplateNumber": 0,
        "typeOfFirstFixedSurface": 106,
        "typeOfSecondFixedSurface": 106,
        "scaledValueOfFirstFixedSurface": 10,
    },
)

In [ ]:
examples[470] = (
    {"parameter": "cedarkit.sw-40-100"},
    {
        "discipline": 0,
        "parameterCategory": 1,
        "parameterNumber": 0,
        "productDefinitionTemplateNumber": 0,
        "typeOfFirstFixedSurface": 106,
        "typeOfSecondFixedSurface": 106,
        "scaledValueOfFirstFixedSurface": 40,
    },
)

In [ ]:
examples[471] = (
    {"parameter": "cedarkit.sw-100-200"},
    {
        "discipline": 0,
        "parameterCategory": 1,
        "parameterNumber": 0,
        "productDefinitionTemplateNumber": 0,
        "typeOfFirstFixedSurface": 106,
        "typeOfSecondFixedSurface": 106,
        "scaledValueOfFirstFixedSurface": 100,
    },
)

## 核对具名查询与原生键

In [ ]:
with from_source({"kind": "file", "path": str(gfs_grib2_orig_file_path)}) as source:
    named = source.sel([query for query, _ in examples.values() if query is not None]).all()
    native = source.sel([{"native": {"grib": keys}} for _, keys in examples.values()]).all()
    fields = {field.metadata["source"]["ordinal"]: field for field in named}
    native_fields = {field.metadata["source"]["ordinal"]: field for field in native}
    assert native_fields.keys() == examples.keys()
    assert fields.keys() == {ordinal for ordinal, (query, _) in examples.items() if query is not None}

    for ordinal, field in fields.items():
        by_native = native_fields[ordinal]
        assert field.identity.parameter == examples[ordinal][0]["parameter"]
        assert field.metadata["source"] == by_native.metadata["source"]
        assert field.read_grib_message() == by_native.read_grib_message()

    with gfs_grib2_orig_file_path.open("rb") as original:
        for field in native_fields.values():
            location = field.metadata["source"]
            original.seek(location["offset"])
            assert field.read_grib_message() == original.read(location["length"])